# HireFlow 系统评估

Run All Cells → Pipeline → 指标 → 报告 → 对比 → Agent 测试

In [1]:
# Cell 1: 初始化
import sys, os, time, json, glob, math
from datetime import datetime
import pytz
from unittest.mock import patch

project_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "evaluation" else os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

sydney_tz = pytz.timezone("Australia/Sydney")
now = datetime.now(sydney_tz)
report_time = now.strftime("%I-%M-%p")
display_time = now.strftime("%I:%M %p")
report_date = now.strftime("%Y-%m-%d")
report_filename = f"{report_date}-{report_time}"

reports_dir = os.path.join(os.getcwd(), "reports")
os.makedirs(reports_dir, exist_ok=True)
report_md = os.path.join(reports_dir, f"{report_filename}.md")
report_json = os.path.join(reports_dir, f"{report_filename}.json")

print(f"报告: reports/{report_filename}.md + .json")
print(f"时间: {report_date} {display_time}")

报告: reports/2026-06-21-03-41-PM.md + .json
时间: 2026-06-21 03:41 PM


In [2]:
# Cell 2: 健康检查 (容错: 服务未启动不会中断)
from app.utils.config import settings
health = {"llm": False, "embedding": False, "postgres": False, "qdrant": False}

try:
    from openai import OpenAI
    c = OpenAI(base_url=settings.llm.local_base_url, api_key=settings.llm.local_api_key, timeout=5)
    models = [m.id for m in c.models.list().data]
    if settings.llm.local_model in models:
        health["llm"] = True
except Exception as e:
    pass

try:
    c = OpenAI(base_url=settings.embedding.local_base_url, api_key=settings.embedding.local_api_key, timeout=5)
    emb_dim = len(c.embeddings.create(model=settings.embedding.local_model, input="t").data[0].embedding)
    health["embedding"] = True
except Exception:
    emb_dim = settings.embedding.dimension

try:
    from app.database.session import init_db; init_db()
    health["postgres"] = True
except Exception:
    pass

try:
    from qdrant_client import QdrantClient
    QdrantClient(url=settings.qdrant.url, timeout=5).get_collections()
    health["qdrant"] = True
except Exception:
    pass

h = health
print(f"LLM={'✅' if h['llm'] else '⚠️'} Embed={'✅' if h['embedding'] else '⚠️'} PG={'✅' if h['postgres'] else '⚠️'} Qdrant={'✅' if h['qdrant'] else '⚠️'}")
if not h["llm"]:
    print("  LLM未启动: Cell 3 (Pipeline) 将跳过, Cell 6-8 (Agent测试) 不受影响")

LLM=✅ Embed=✅ PG=⚠️ Qdrant=⚠️


In [3]:
# Cell 3: Pipeline 运行 (需LLM; 未启动则跳过)
pipeline_ran = False
if health["llm"]:
    from app.agents.jd_agent import analyze_jd
    from app.agents.resume_agent import batch_parse_resumes
    from app.agents.match_agent import batch_match_candidates
    from app.agents.ranking_agent import rank_candidates

    TEST_JD = """
岗位名称: Python 后端开发工程师
必备技能: Python, FastAPI, PostgreSQL, Docker, Git
加分技能: LangChain, RAG, Redis
学历要求: 计算机相关专业本科及以上
经验要求: 0-3年
"""
    TEST_RESUMES = {
        "E001": "姓名: 张工\\n技能: Python,FastAPI,PostgreSQL,Docker,Git,Redis\\n教育: 2020-2024 北大 CS学士\\n经历: 2023 Python实习生",
        "E002": "姓名: 李工\\n技能: Python,Django,MySQL,Docker,Git\\n教育: 2019-2023 浙大 SE学士\\n经历: 2022 Django实习生",
        "E003": "姓名: 王工\\n技能: Python,FastAPI,PostgreSQL,LangChain,RAG,Docker,Git\\n教育: 2021-2023 清华 AI硕士\\n经历: 2023 AI后端实习生",
    }

    print("运行 Pipeline...\n")
    timeline = []
    total_start = time.time()

    t0 = time.time()
    jd_profile = await analyze_jd(TEST_JD)
    t = time.time() - t0
    timeline.append(("JD 解析", t))
    print(f"  [1/4] JD解析: {t:.1f}s")

    t0 = time.time()
    profiles = await batch_parse_resumes(TEST_RESUMES)
    t = time.time() - t0
    timeline.append(("简历解析", t))
    ids = list(TEST_RESUMES.keys())
    for i, p in enumerate(profiles): p["candidate_id"] = ids[i]
    print(f"  [2/4] 简历解析: {t:.1f}s ({len(profiles)}份)")

    t0 = time.time()
    rubric = jd_profile.pop("rubric", None)
    matches = await batch_match_candidates(jd_profile, profiles, rubric=rubric)
    t = time.time() - t0
    timeline.append(("匹配评分", t))
    print(f"  [3/4] 匹配评分: {t:.1f}s")

    t0 = time.time()
    ranking = await rank_candidates(matches)
    t = time.time() - t0
    timeline.append(("排序", t))
    print(f"  [4/4] 排序: {t:.1f}s")

    total_time = time.time() - total_start
    ranked = ranking.get("ranked_candidates", [])
    scores = [c.get("total_score", 0) for c in ranked]
    pipeline_ran = True
    print(f"\nPipeline 完成 | 总耗时: {total_time:.1f}s\n")
else:
    print("Pipeline 跳过 (LLM未启动)。Agent 测试 Cell 6-8 可正常运行。")

运行 Pipeline...



/Users/chris/miniconda3/envs/hireflowagents/lib/python3.11/site-packages/qdrant_client/qdrant_remote.py:290: UserWarning: Failed to obtain server version. Unable to check client-server compatibility. Set check_compatibility=False to skip version check.
  show_warning(


  [1/4] JD解析: 6.8s


  [2/4] 简历解析: 11.6s (3份)


  [3/4] 匹配评分: 17.6s


  [4/4] 排序: 2.1s

Pipeline 完成 | 总耗时: 38.1s



In [4]:
# Cell 4: 指标 + 报告 (仅当 Pipeline 已运行)
if pipeline_ran:
    jd_skills = jd_profile.get("required_skills", [])
    relevance = {}
    for cid, text in TEST_RESUMES.items():
        hits = sum(1 for s in jd_skills if s.lower() in text.lower())
        relevance[cid] = hits / max(len(jd_skills), 1)

    ranked_ids = [c.get("candidate_id", "?") for c in ranked]

    def p_at_k(rids, rel, k, thr=0.5):
        top = rids[:k]
        return sum(1 for c in top if rel.get(c,0) >= thr) / k if top else 0
    def ndcg(rids, rel, k):
        if not rids: return 0
        dcg = sum(rel.get(c,0)/math.log2(i+2) for i,c in enumerate(rids[:k]))
        ideal = sorted(rel.keys(),key=lambda x:rel.get(x,0),reverse=True)
        idcg = sum(rel.get(c,0)/math.log2(i+2) for i,c in enumerate(ideal[:k]))
        return dcg/idcg if idcg>0 else 0

    metrics = {f"precision_at_{k}": round(p_at_k(ranked_ids,relevance,k),3) for k in [1,2,3]}
    metrics["ndcg_at_3"] = round(ndcg(ranked_ids,relevance,3),3)
    if scores:
        metrics["score_max"] = max(scores)
        metrics["score_min"] = min(scores)
        metrics["score_mean"] = round(sum(scores)/len(scores),1)
        metrics["score_range"] = max(scores)-min(scores)

    # JSON
    ed = {"timestamp":datetime.now(sydney_tz).isoformat(),"pipeline":{"steps":[{"step":n,"time":round(t,2)} for n,t in timeline],"total_time":round(total_time,2)},"ranking":ranked_ids,"scores":[{"id":c.get("candidate_id"),"score":c.get("total_score",0),"rec":c.get("recommendation","")} for c in ranked],"metrics":metrics}
    with open(report_json,"w") as f: json.dump(ed,f,ensure_ascii=False,indent=2)

    print(f"报告: reports/{report_filename}.md + .json")
    print(f"指标: P@1={metrics['precision_at_1']} P@3={metrics['precision_at_3']} NDCG@3={metrics['ndcg_at_3']} 均值={metrics['score_mean']}")
else:
    print("跳过 (Pipeline 未运行)")

报告: reports/2026-06-21-03-41-PM.md + .json
指标: P@1=1.0 P@3=1.0 NDCG@3=0.965 均值=-5.7


In [5]:
# Cell 5: 历史对比
json_files = sorted(glob.glob(os.path.join(reports_dir, "*.json")), reverse=True)
if len(json_files) == 0:
    print("暂无历史数据")
elif len(json_files) == 1:
    print("仅 1 次记录, 无法对比")
else:
    recent = json_files[:5]
    records = [json.load(open(f)) for f in recent]
    print(f"历史对比 (最近 {len(records)} 次, 共 {len(json_files)} 次)\n")
    print("Pipeline 耗时趋势:")
    for i, rec in enumerate(records):
        ts = rec.get("timestamp","?")[:16].replace("T"," ")
        total = rec["pipeline"]["total_time"]
        delta = "(当前)" if i == 0 else (f"↓{records[i-1]['pipeline']['total_time']-total:.1f}s" if total < records[i-1]['pipeline']['total_time'] else f"↑{total-records[i-1]['pipeline']['total_time']:.1f}s")
        print(f"  {ts}  {total:.1f}s  {delta}")
    print("\n指标趋势:")
    keys = [("ndcg_at_3","NDCG@3"),("score_mean","平均分")]
    for key, label in keys:
        newest = records[0]["metrics"].get(key,0)
        oldest = records[-1]["metrics"].get(key,0)
        diff = newest - oldest
        trend = "↗" if diff > 0.01 else ("↘" if diff < -0.01 else "→")
        print(f"  {label}: {oldest:.2f} → {newest:.2f} {trend}")

历史对比 (最近 3 次, 共 3 次)

Pipeline 耗时趋势:
  2026-06-21 15:42  38.1s  (当前)
  2026-05-31 16:35  49.7s  ↑11.6s
  2026-05-31 16:33  47.3s  ↓2.4s

指标趋势:
  NDCG@3: 1.00 → 0.96 ↘
  平均分: 81.30 → -5.70 ↘


In [6]:
# Cell 6: Interview Agent 测试 (mock, 无需 LLM)
from app.agents.interview_agent import generate_questions, _build_question_prompt

print("【Interview Agent】\n")
ok = 0

prompt = _build_question_prompt(
    jd_profile={"job_title":"Python后端","required_skills":["Python","FastAPI"],"technical_requirements":["Docker"],"soft_skills":["沟通"]},
    candidate_profile={"name":"测试","skills":["Python"],"projects":[{"name":"API项目","description":"FastAPI后端","technologies":["FastAPI"]}],"work_experience":[]},
    match_result={"risks":["经验不足"],"total_score":75,"recommendation":"Medium"},
)
assert "Python后端" in prompt and "API项目" in prompt and "经验不足" in prompt
ok += 1; print(f"  1. 提示词 OK")

with patch("app.agents.interview_agent.call_llm") as m:
    m.return_value = '[{"question_type":"technical","question":"什么是FastAPI?","purpose":"测试技术"}]'
    qs = await generate_questions(
        jd_profile={"job_title":"测试","required_skills":["Python"],"technical_requirements":[],"soft_skills":[]},
        candidate_profile={"name":"张三","skills":["Python"],"projects":[],"work_experience":[]},
        match_result={"risks":[],"total_score":80,"recommendation":"Strong"},
    )
assert isinstance(qs, list) and len(qs) >= 1
assert "question" in qs[0] and "purpose" in qs[0]
ok += 1; print(f"  2. 问题结构 OK ({len(qs)}个)")

with patch("app.agents.interview_agent.call_llm") as m:
    m.return_value = "无效JSON"
    qs = await generate_questions(
        jd_profile={"job_title":"测试","required_skills":["Python"],"technical_requirements":[],"soft_skills":[]},
        candidate_profile={"name":"张三","skills":["Python"],"projects":[],"work_experience":[]},
        match_result={"risks":[],"total_score":80,"recommendation":"Strong"},
    )
assert len(qs) >= 1
ok += 1; print(f"  3. 回退 OK")

print(f"\n  Interview: {ok}/3 通过")

【Interview Agent】

  1. 提示词 OK
  2. 问题结构 OK (1个)
  3. 回退 OK

  Interview: 3/3 通过


In [7]:
# Cell 7: Evaluation Agent 测试 (mock)
from app.agents.evaluation_agent import evaluate_candidate, _build_eval_prompt

print("【Evaluation Agent】\n")
ok = 0

prompt = _build_eval_prompt(
    interview_feedback="候选人技术问答表现优异",
    candidate_profile={"name":"张三","skills":["Python"]},
    match_result={"risks":["部署经验不足"],"total_score":80,"recommendation":"Strong"},
    jd_profile={"job_title":"Python后端"},
)
assert "候选人技术问答表现优异" in prompt
ok += 1; print(f"  1. 提示词 OK")

with patch("app.agents.evaluation_agent.call_llm") as m:
    m.return_value = '{"technical_depth_score":8,"communication_score":7,"problem_solving_score":6,"risk_resolution":[{"risk":"经验不足","status":"resolved","reason":"表现出色"}],"strengths":["表达清晰"],"concerns":[],"summary":"良好","recommendation":"Recommend"}'
    r = await evaluate_candidate(
        interview_feedback="表现良好",
        candidate_profile={"name":"张三","skills":["Python"]},
        match_result={"risks":["经验不足"],"total_score":80},
        jd_profile={"job_title":"Python后端"},
    )
assert r["requires_human_review"] is True
assert "recommendation" in r and "strengths" in r
ok += 1; print(f"  2. 结构 OK (recommendation={r['recommendation']})")

with patch("app.agents.evaluation_agent.call_llm") as m:
    m.return_value = "无效"
    r = await evaluate_candidate(
        interview_feedback="简短", candidate_profile={"name":"张三","skills":["Python"]},
        match_result={"risks":[],"total_score":80}, jd_profile={"job_title":"测试"},
    )
assert r["requires_human_review"] and r["recommendation"] == "Hold"
ok += 1; print(f"  3. 回退 OK")

print(f"\n  Evaluation: {ok}/3 通过")

【Evaluation Agent】

  1. 提示词 OK
  2. 结构 OK (recommendation=Recommend)
  3. 回退 OK

  Evaluation: 3/3 通过


In [8]:
# Cell 8: Email Agent 测试 (mock)
from app.agents.email_agent import generate_email_draft

print("【Email Agent】\n")
ok = 0

with patch("app.agents.email_agent.call_llm") as m:
    m.return_value = '{"subject":"面试邀请","body":"尊敬的张三..."}'
    d = await generate_email_draft(
        candidate_profile={"name":"张三","skills":["Python"]},
        job_title="Python后端", email_type="interview_invite",
    )
assert d["status"] == "draft" and d["requires_human_approval"] is True
ok += 1; print(f"  1. 草稿结构 OK")

with patch("app.agents.email_agent.call_llm") as m:
    m.return_value = '{"subject":"感谢申请","body":"我们很遗憾..."}'
    d = await generate_email_draft(
        candidate_profile={"name":"张三"}, job_title="岗位",
        email_type="rejection",
        evaluation_result={"recommendation":"Not Recommend"},
    )
assert "Not Recommend" not in d["body"]
ok += 1; print(f"  2. 拒信礼貌 OK")

for et in ["interview_invite","rejection","follow_up","next_round"]:
    with patch("app.agents.email_agent.call_llm") as m:
        m.return_value = f'{{"subject":"{et}","body":"内容"}}'
        d = await generate_email_draft(
            candidate_profile={"name":"测试"}, job_title="岗位", email_type=et,
        )
    assert d["email_type"] == et and d["status"] == "draft"
ok += 1; print(f"  3. 4种类型 OK")

with patch("app.agents.email_agent.call_llm") as m:
    m.return_value = "无效JSON"
    d = await generate_email_draft(
        candidate_profile={"name":"张三"}, job_title="测试", email_type="interview_invite",
    )
assert d["status"] == "draft" and len(d["body"]) > 0
ok += 1; print(f"  4. 回退 OK")

print(f"\n  Email: {ok}/4 通过")
print(f"\n{'='*40}")
print(f"  新 Agent 测试: Interview(3) + Evaluation(3) + Email(4) = 全通过")
print(f"{'='*40}")

【Email Agent】

  1. 草稿结构 OK
  2. 拒信礼貌 OK
  3. 4种类型 OK
  4. 回退 OK

  Email: 4/4 通过

  新 Agent 测试: Interview(3) + Evaluation(3) + Email(4) = 全通过


### 使用说明

**前置条件 (需提前在终端启动):**
```bash
# 终端 1: 数据库
docker compose up -d postgres qdrant
# 终端 2 / LM Studio: 加载 hermes-3 + embedding 模型
```

**运行:** `conda activate hireflowagents && cd evaluation && jupyter notebook 系统评估报告.ipynb`

**Cell 说明:**
| Cell | 内容 | 需要外部服务 |
|---|---|---|
| 1 | 初始化 | 否 |
| 2 | 健康检查 (容错) | 否 |
| 3 | Pipeline 运行 | LLM |
| 4 | 指标 + 报告 | 否 |
| 5 | 历史对比 | 否 |
| 6 | Interview Agent 测试 | 否 (mock) |
| 7 | Evaluation Agent 测试 | 否 (mock) |
| 8 | Email Agent 测试 | 否 (mock) |